In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# -----------------------------
# 1. Hyperparameters
# -----------------------------
INPUT_DIM = 15      # vocabulary size for input tokens
OUTPUT_DIM = 15     # vocabulary size for output tokens
EMB_DIM = 16
HID_DIM = 32
NUM_LAYERS = 1
LEARNING_RATE = 0.01
EPOCHS = 300

SOS_TOKEN = 1
EOS_TOKEN = 2
PAD_TOKEN = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# 2. Encoder
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_TOKEN)
        self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True)

    def forward(self, src):
        # src shape: [batch_size, src_len]
        embedded = self.embedding(src)              # [batch_size, src_len, emb_dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell


# -----------------------------
# 3. Decoder
# -----------------------------
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_TOKEN)
        self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, x, hidden, cell):
        # x shape: [batch_size]
        x = x.unsqueeze(1)                          # [batch_size, 1]
        embedded = self.embedding(x)               # [batch_size, 1, emb_dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))  # [batch_size, output_dim]
        return prediction, hidden, cell


# -----------------------------
# 4. Seq2Seq Model
# -----------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # trg: [batch_size, trg_len]
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = OUTPUT_DIM

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(device)

        hidden, cell = self.encoder(src)

        # first input to decoder is SOS token
        x = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(x, hidden, cell)
            outputs[:, t, :] = output

            best_guess = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_ratio
            x = trg[:, t] if teacher_force else best_guess

        return outputs


# -----------------------------
# 5. Create Toy Dataset
# -----------------------------
def generate_sample(seq_len=5):
    # tokens from 3 to 14; 0=PAD, 1=SOS, 2=EOS
    seq = [random.randint(3, 14) for _ in range(seq_len)]
    src = seq
    trg = [SOS_TOKEN] + list(reversed(seq)) + [EOS_TOKEN]
    return src, trg

data = [generate_sample(seq_len=5) for _ in range(1000)]


def to_tensor_batch(batch):
    src_batch = [item[0] for item in batch]
    trg_batch = [item[1] for item in batch]

    src_tensor = torch.tensor(src_batch, dtype=torch.long).to(device)
    trg_tensor = torch.tensor(trg_batch, dtype=torch.long).to(device)

    return src_tensor, trg_tensor


# -----------------------------
# 6. Initialize Model
# -----------------------------
encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS)
model = Seq2Seq(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)


# -----------------------------
# 7. Training
# -----------------------------
for epoch in range(EPOCHS):
    model.train()
    batch = random.sample(data, 32)
    src, trg = to_tensor_batch(batch)

    output = model(src, trg)   # [batch_size, trg_len, output_dim]

    # ignore first token for loss
    output_dim = output.shape[-1]
    output = output[:, 1:, :].reshape(-1, output_dim)
    trg = trg[:, 1:].reshape(-1)

    loss = criterion(output, trg)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}")


# -----------------------------
# 8. Inference
# -----------------------------
def predict(model, src_seq, max_len=10):
    model.eval()
    src_tensor = torch.tensor([src_seq], dtype=torch.long).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    x = torch.tensor([SOS_TOKEN], dtype=torch.long).to(device)
    outputs = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(x, hidden, cell)

        best_token = output.argmax(1).item()

        if best_token == EOS_TOKEN:
            break

        outputs.append(best_token)
        x = torch.tensor([best_token], dtype=torch.long).to(device)

    return outputs


# -----------------------------
# 9. Test
# -----------------------------
test_seq = [10, 4, 6, 7, 8]
prediction = predict(model, test_seq)

print("\nInput sequence:     ", test_seq)
print("Expected output:    ", list(reversed(test_seq)))
print("Model prediction:   ", prediction)

Epoch [50/300], Loss: 1.5128
Epoch [100/300], Loss: 0.9475
Epoch [150/300], Loss: 0.4920
Epoch [200/300], Loss: 0.4129
Epoch [250/300], Loss: 0.2813
Epoch [300/300], Loss: 0.1525

Input sequence:      [10, 4, 6, 7, 8]
Expected output:     [8, 7, 6, 4, 10]
Model prediction:    [8, 7, 6, 4, 10]
